# Statistical Inference Lab & SI Project: Airbnb Rental Price Prediction

**Project:** Regression: Airbnb Rental Price Prediction  
**Environment:** Google Colab / Jupyter / VS Code  
**Dataset:** `data/airbnb_listings.csv` or `data/sample_listings.csv` from the existing project

This notebook combines the practical topics with the main Airbnb rental price regression project. It uses the real CSV available in the project folder and does not invent model results.

## How to use in the existing project
Save this notebook as `airbnb_practicals.ipynb` inside `D:\SI!\airbnb_price_app\`. The notebook searches the `data/` folder relative to its location. Run cells from top to bottom. If a package is missing, install it in the same Python environment used by your project.

---
## Setup: Imports, paths, and dataset loading

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
# If opened from a different working directory, find the project folder by checking common paths.
if not (ROOT / "data").exists() and (ROOT / "airbnb_price_app" / "data").exists():
    ROOT = ROOT / "airbnb_price_app"
DATA_DIR = ROOT / "data"
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

def load_airbnb_data():
    candidates = [
        DATA_DIR / "airbnb_listings.csv",
        DATA_DIR / "sample_listings.csv",
    ]
    for path in candidates:
        if path.exists():
            return pd.read_csv(path, low_memory=False), path
    csvs = sorted(DATA_DIR.glob("*.csv")) if DATA_DIR.exists() else []
    if csvs:
        return pd.read_csv(csvs[0], low_memory=False), csvs[0]
    raise FileNotFoundError(
        f"No CSV found in {DATA_DIR}. Add airbnb_listings.csv or sample_listings.csv."
    )

def clean_price(series):
    return pd.to_numeric(series.astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce")

df, dataset_path = load_airbnb_data()
df.columns = df.columns.astype(str).str.strip()
print("Dataset:", dataset_path)
print("Shape:", df.shape)
display(df.head())

Dataset: d:\SI!\airbnb_price_app\data\airbnb_listings.csv
Shape: (6040, 14)


,id,name,neighbourhood,room_type,bedrooms,beds,accommodates,bathrooms,minimum_nights,number_of_reviews,availability_365,weekday_price,weekend_price,price
0,105530,Private room in Suburb North,Suburb North,Private room,1.0,1.0,1,1.5,1,10.0,57,33.57,34.61,33.0
1,100711,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,3.0,4,2.0,30,218.0,207,83.82,86.30,80.0
2,104924,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,NaN,5,2.0,2,25.0,0,98.93,107.42,95.0
3,102154,Private room in Suburb North,Suburb North,Private room,1.0,2.0,1,1.5,2,33.0,198,27.49,27.17,27.0
4,101273,Entire home/apt in Midtown,Midtown,Entire home/apt,3.0,4.0,7,2.5,1,157.0,59,174.87,199.51,167.0


---
## Practical 1: Web scraping fundamentals using Requests and BeautifulSoup

**Objective:** Make an HTTP request, check the response, parse HTML, and extract page title and links. Use only a website that permits educational scraping. Airbnb's terms and robots rules may restrict automated scraping, so this demo uses `example.com` unless you provide an approved URL.

In [2]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urlparse, urljoin
from urllib.robotparser import RobotFileParser

def robots_allows(url):
    parts = urlparse(url)
    parser = RobotFileParser()
    parser.set_url(f"{parts.scheme}://{parts.netloc}/robots.txt")
    try:
        parser.read()
        return parser.can_fetch("*", url)
    except Exception:
        return False

scrape_url = "https://example.com/"
print("Robots permission confirmed:", robots_allows(scrape_url))
if robots_allows(scrape_url):
    response = requests.get(scrape_url, timeout=15, headers={"User-Agent": "EducationalProject/1.0"})
    print("HTTP status:", response.status_code)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")
    print("Title:", soup.title.get_text(strip=True) if soup.title else "No title")
    links = [{"text": a.get_text(" ", strip=True), "url": urljoin(scrape_url, a["href"])}
             for a in soup.select("a[href]")]
    display(pd.DataFrame(links).head(10))
else:
    print("Scraping skipped: permission was not confirmed.")

Robots permission confirmed: True
HTTP status: 200
Title: Example Domain


""


---
## Practical 2: Dataset collection, inspection, and data cleaning

**Objective:** Inspect the Airbnb dataset, check data types and missing values, remove duplicates, and clean the nightly price target when the `price` column exists.

In [3]:
print("Source:", dataset_path)
print("Rows, columns:", df.shape)
display(df.head())
display(df.dtypes.rename("dtype").to_frame())
missing_summary = df.isna().sum().sort_values(ascending=False).rename("missing_count").to_frame()
display(missing_summary.head(25))
print("Duplicate rows:", int(df.duplicated().sum()))

clean_df = df.drop_duplicates().copy()
if "price" in clean_df.columns:
    clean_df["price"] = clean_price(clean_df["price"])
    clean_df = clean_df[clean_df["price"].notna() & (clean_df["price"] > 0)]
clean_path = OUTPUT_DIR / "cleaned_listings.csv"
clean_df.to_csv(clean_path, index=False)
print("Cleaned shape:", clean_df.shape)
print("Saved:", clean_path)
display(clean_df.head())

Source: d:\SI!\airbnb_price_app\data\airbnb_listings.csv
Rows, columns: (6040, 14)


,id,name,neighbourhood,room_type,bedrooms,beds,accommodates,bathrooms,minimum_nights,number_of_reviews,availability_365,weekday_price,weekend_price,price
0,105530,Private room in Suburb North,Suburb North,Private room,1.0,1.0,1,1.5,1,10.0,57,33.57,34.61,33.0
1,100711,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,3.0,4,2.0,30,218.0,207,83.82,86.30,80.0
2,104924,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,NaN,5,2.0,2,25.0,0,98.93,107.42,95.0
3,102154,Private room in Suburb North,Suburb North,Private room,1.0,2.0,1,1.5,2,33.0,198,27.49,27.17,27.0
4,101273,Entire home/apt in Midtown,Midtown,Entire home/apt,3.0,4.0,7,2.5,1,157.0,59,174.87,199.51,167.0


,dtype
id,int64
name,str
neighbourhood,str
room_type,str
bedrooms,float64
beds,float64
accommodates,int64
bathrooms,float64
minimum_nights,int64
number_of_reviews,float64


,missing_count
bathrooms,239
bedrooms,195
beds,193
number_of_reviews,117
name,0
id,0
neighbourhood,0
room_type,0
accommodates,0
minimum_nights,0


Duplicate rows: 40
Cleaned shape: (5995, 14)
Saved: d:\SI!\airbnb_price_app\outputs\cleaned_listings.csv


,id,name,neighbourhood,room_type,bedrooms,beds,accommodates,bathrooms,minimum_nights,number_of_reviews,availability_365,weekday_price,weekend_price,price
0,105530,Private room in Suburb North,Suburb North,Private room,1.0,1.0,1,1.5,1,10.0,57,33.57,34.61,33.0
1,100711,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,3.0,4,2.0,30,218.0,207,83.82,86.30,80.0
2,104924,Entire home/apt in Suburb South,Suburb South,Entire home/apt,2.0,NaN,5,2.0,2,25.0,0,98.93,107.42,95.0
3,102154,Private room in Suburb North,Suburb North,Private room,1.0,2.0,1,1.5,2,33.0,198,27.49,27.17,27.0
4,101273,Entire home/apt in Midtown,Midtown,Entire home/apt,3.0,4.0,7,2.5,1,157.0,59,174.87,199.51,167.0


---
## Practical 3: Exploratory Data Analysis (EDA) and visualization

**Objective:** Generate descriptive statistics, numeric distributions, price distribution, and a correlation heatmap where the relevant columns exist.

In [ ]:
display(clean_df.describe(include="all").T)
numeric_df = clean_df.select_dtypes(include="number")
if not numeric_df.empty:
    numeric_df.hist(figsize=(14, 9), bins=30)
    plt.tight_layout()
    plt.show()

if "price" in clean_df.columns:
    price_values = clean_price(clean_df["price"]).dropna()
    price_values = price_values[price_values > 0]
    if not price_values.empty:
        plt.figure(figsize=(9, 5))
        plt.hist(price_values, bins=40)
        plt.title("Airbnb Nightly Price Distribution")
        plt.xlabel("Price (dataset currency)")
        plt.ylabel("Listing count")
        plt.show()

if numeric_df.shape[1] >= 2:
    plt.figure(figsize=(12, 8))
    plt.imshow(numeric_df.corr(), vmin=-1, vmax=1, aspect="auto")
    plt.colorbar(label="Pearson correlation")
    plt.xticks(range(len(numeric_df.columns)), numeric_df.columns, rotation=90)
    plt.yticks(range(len(numeric_df.columns)), numeric_df.columns)
    plt.title("Numeric Feature Correlation")
    plt.tight_layout()
    plt.show()

---
## Practical 4: Probability distributions and Central Limit Theorem

**Objective:** Fit a normal curve to valid nightly prices and demonstrate the Central Limit Theorem using repeated sample means. The KS p-value shown is only descriptive because the normal parameters are estimated from the same data.

In [ ]:
from scipy import stats

if "price" not in clean_df.columns:
    print("No price column; skip price-distribution analysis until target column is configured.")
else:
    values = clean_price(clean_df["price"]).dropna().to_numpy(dtype=float)
    values = values[np.isfinite(values) & (values > 0)]
    if len(values) >= 2:
        mu = float(values.mean())
        sd = float(values.std(ddof=1))
        print(f"Valid prices: {len(values)} | Mean: {mu:.3f} | Sample SD: {sd:.3f}")
        plt.figure(figsize=(9, 5))
        plt.hist(values, bins=40, density=True, alpha=0.65, label="Observed prices")
        if sd > 0:
            grid = np.linspace(values.min(), values.max(), 300)
            plt.plot(grid, stats.norm.pdf(grid, loc=mu, scale=sd), label="Fitted normal curve")
            ks = stats.kstest(values, lambda v: stats.norm.cdf(v, loc=mu, scale=sd))
            print("KS statistic:", ks.statistic, "| p-value:", ks.pvalue)
        plt.legend()
        plt.title("Price Distribution")
        plt.show()

        rng = np.random.default_rng(42)
        sample_size = min(30, len(values))
        sample_means = [rng.choice(values, size=sample_size, replace=True).mean() for _ in range(1000)]
        plt.figure(figsize=(9, 5))
        plt.hist(sample_means, bins=35)
        plt.title(f"CLT: Distribution of 1,000 Sample Means (n={sample_size})")
        plt.xlabel("Sample mean price")
        plt.show()
    else:
        print("Not enough valid price values.")

---
## Practical 5: Descriptive statistics and confidence interval

**Objective:** Calculate mean, median, mode, range, variance, standard deviation, IQR, and a 95% t confidence interval for numerical columns.

In [ ]:
from scipy import stats

for column in clean_df.select_dtypes(include="number").columns:
    values = pd.to_numeric(clean_df[column], errors="coerce").dropna()
    if len(values) == 0:
        continue
    q1, q3 = values.quantile([0.25, 0.75])
    print(f"\nColumn: {column}")
    print("Mean:", values.mean())
    print("Median:", values.median())
    print("Mode:", values.mode().tolist()[:5])
    print("Range:", values.max() - values.min())
    print("Variance:", values.var())
    print("Standard deviation:", values.std())
    print("IQR:", q3 - q1)
    if len(values) > 1:
        ci = stats.t.interval(0.95, df=len(values)-1, loc=values.mean(), scale=stats.sem(values))
        print("95% confidence interval for mean:", ci)

---
## Practical 6: Hypothesis testing and correlation analysis

**Objective:** Demonstrate independent t-test, paired t-test, one-way ANOVA, chi-square test, and calculate Pearson/Spearman correlations on available numeric Airbnb features. The small arrays below are labelled teaching examples, not Airbnb-derived evidence.

In [ ]:
# Teaching examples for common hypothesis tests
group_a = np.array([45, 50, 55, 60, 50])
group_b = np.array([90, 82, 68, 43, 87])
t_stat, p_value = stats.ttest_ind(group_a, group_b, equal_var=False)
print(f"Independent Welch t-test: t={t_stat:.4f}, p={p_value:.4f}")

before = np.array([67, 81, 61, 44, 49])
after = np.array([87, 96, 59, 49, 58])
t_stat, p_value = stats.ttest_rel(before, after)
print(f"Paired t-test: t={t_stat:.4f}, p={p_value:.4f}")

f_stat, p_value = stats.f_oneway([12,14,15,13,16], [18,17,19,20,16], [10,11,12,9,13])
print(f"One-way ANOVA: F={f_stat:.4f}, p={p_value:.4f}")

chi2, p_value, dof, expected = stats.chi2_contingency([[20, 15], [10, 25]])
print(f"Chi-square test: chi2={chi2:.4f}, p={p_value:.4f}, df={dof}")

numeric_df = clean_df.select_dtypes(include="number")
if numeric_df.shape[1] >= 2:
    print("Pearson correlation:")
    display(numeric_df.corr(method="pearson"))
    print("Spearman correlation:")
    display(numeric_df.corr(method="spearman"))

---
## Practical 7: Scaling and simple linear regression

**Objective:** Demonstrate StandardScaler and MinMaxScaler, then train a simple linear regression model to predict `price` from one available numeric feature. The feature is selected from common Airbnb columns.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

demo_values = np.array([[10000], [25000], [50000], [75000], [100000]], dtype=float)
print("MinMax scaled example:\n", MinMaxScaler().fit_transform(demo_values).ravel())
print("Standard scaled example:\n", StandardScaler().fit_transform(demo_values).ravel())

if "price" not in clean_df.columns:
    print("Regression skipped: target column 'price' not found.")
else:
    candidate_features = [c for c in ["accommodates", "bedrooms", "beds", "minimum_nights", "number_of_reviews"]
                          if c in clean_df.columns]
    if not candidate_features:
        candidate_features = [c for c in clean_df.select_dtypes(include="number").columns if c != "price"]
    if not candidate_features:
        print("Regression skipped: no numeric predictor column found.")
    else:
        feature = candidate_features[0]
        work = clean_df[[feature, "price"]].copy()
        work[feature] = pd.to_numeric(work[feature], errors="coerce")
        work["price"] = clean_price(work["price"])
        work = work.replace([np.inf, -np.inf], np.nan).dropna()
        work = work[work["price"] > 0]
        if len(work) < 10:
            print("Regression skipped: fewer than 10 valid rows.")
        else:
            X = work[[feature]]
            y = work["price"]
            X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
            scaler = StandardScaler()
            X_train_s = scaler.fit_transform(X_train)
            X_test_s = scaler.transform(X_test)
            model = LinearRegression().fit(X_train_s, y_train)
            pred = model.predict(X_test_s)
            print("Predictor:", feature)
            print("MAE:", mean_absolute_error(y_test, pred))
            print("RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
            print("R2:", r2_score(y_test, pred))
            plt.figure(figsize=(7, 6))
            plt.scatter(y_test, pred, alpha=0.5)
            low, high = min(y_test.min(), pred.min()), max(y_test.max(), pred.max())
            plt.plot([low, high], [low, high], "--")
            plt.xlabel("Actual price")
            plt.ylabel("Predicted price")
            plt.title("Simple Linear Regression: Actual vs Predicted")
            plt.show()

---
## Practical 8: Regression model comparison

**Objective:** Compare Linear Regression, Ridge, Lasso, Decision Tree, and Random Forest with the same train/test split and preprocessing pipeline. The target is the dataset's nightly `price` column. Models are evaluated using MAE, MSE, RMSE, and R².

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

if "price" not in clean_df.columns:
    print("Model comparison skipped: target column 'price' not found.")
else:
    model_df = clean_df.drop_duplicates().copy()
    model_df["price"] = clean_price(model_df["price"])
    model_df = model_df[model_df["price"].notna() & (model_df["price"] > 0)]
    target = "price"
    excluded = [c for c in ["price", "id", "listing_id", "scrape_id", "name",
                             "description", "host_name", "host_about"] if c in model_df.columns]
    X = model_df.drop(columns=excluded).dropna(axis=1, how="all")
    y = model_df[target]
    high_cardinality = [c for c in X.select_dtypes(exclude="number").columns
                        if X[c].nunique(dropna=True) > min(500, max(100, len(X)//3))]
    X = X.drop(columns=high_cardinality, errors="ignore")

    if len(X) < 20 or X.shape[1] == 0:
        print("Model comparison skipped: need at least 20 rows and one usable feature.")
    else:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
        numeric_cols = X_train.select_dtypes(include="number").columns.tolist()
        categorical_cols = X_train.select_dtypes(exclude="number").columns.tolist()
        transformers = []
        if numeric_cols:
            transformers.append(("numeric", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ]), numeric_cols))
        if categorical_cols:
            transformers.append(("categorical", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore", min_frequency=2))
            ]), categorical_cols))
        preprocessor = ColumnTransformer(transformers=transformers)

        regressors = {
            "Linear Regression": LinearRegression(),
            "Ridge": Ridge(alpha=1.0),
            "Lasso": Lasso(alpha=0.01, max_iter=5000),
            "Decision Tree": DecisionTreeRegressor(max_depth=15, min_samples_leaf=3, random_state=42),
            "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=20, min_samples_leaf=2,
                                                   n_jobs=-1, random_state=42)
        }
        results = []
        fitted_pipelines = {}
        for name, estimator in regressors.items():
            try:
                pipe = Pipeline([("preprocessor", preprocessor), ("model", estimator)])
                pipe.fit(X_train, y_train)
                pred = pipe.predict(X_test)
                mse = mean_squared_error(y_test, pred)
                results.append({"Model": name, "MAE": mean_absolute_error(y_test, pred),
                                "MSE": mse, "RMSE": np.sqrt(mse), "R2": r2_score(y_test, pred)})
                fitted_pipelines[name] = pipe
            except Exception as exc:
                print(f"Could not fit {name}: {exc}")
        if results:
            results_df = pd.DataFrame(results).sort_values("RMSE")
            display(results_df)
            results_path = OUTPUT_DIR / "airbnb_model_comparison.csv"
            results_df.to_csv(results_path, index=False)
            print("Metrics saved to:", results_path)
            best_name = results_df.iloc[0]["Model"]
            best_model = fitted_pipelines[best_name]
            print("Lowest holdout RMSE model:", best_name)
            import joblib
            model_path = ROOT / "models" / "airbnb_notebook_best_pipeline.joblib"
            model_path.parent.mkdir(exist_ok=True)
            joblib.dump(best_model, model_path)
            print("Saved fitted pipeline:", model_path)
        else:
            print("No models fitted successfully; check dataset features and dependencies.")

---
## Main Project Summary: Regression — Airbnb Rental Price Prediction

### Methodology
1. Load the real listings CSV from the existing project's `data/` folder.
2. Clean duplicate rows and parse the nightly `price` target.
3. Explore distributions, missing values, numeric relationships, and descriptive statistics.
4. Train regression models using a consistent 80/20 holdout split.
5. Compare MAE, MSE, RMSE, and R²; save the best pipeline and metrics.

### Important notes
- Verify the exact price column and currency in your dataset documentation.
- This notebook does not generate fake Airbnb listings or fake accuracy values.
- The standard KS-test p-value is not formally calibrated when normal parameters are estimated from the same sample.
- Use a dataset you are legally permitted to use. Scrape only pages whose terms and robots policy allow it.
- The notebook saves generated files to `outputs/` and the fitted model to `models/`.